# EXP-011 — LR cân bằng lớp

## 01 — Thông tin
Hồ sơ theo kế hoạch chung EXP-001–020. Phân tích kết quả chạy lại đã lưu; ngày đồng bộ 19/11/2026 không phải ngày huấn luyện. Mã nguồn cũ `THAI-EXP-003` chỉ dùng truy vết.

In [1]:
from pathlib import Path
import json
import hashlib
import yaml
import pandas as pd
from IPython.display import display, Image

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/wesad/models.py").is_file())
CONFIG_PATH = ROOT / 'experiments/03_balancing/EXP-011_LR/config.yaml'
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
RESULTS_PATH = ROOT / 'experiments/03_balancing/EXP-011_LR/results.csv'
FOLDER = CONFIG_PATH.parent
PROVENANCE = json.loads((FOLDER / "provenance.json").read_text(encoding="utf-8"))

## 02 — Câu hỏi và giả thuyết
Chỉ thêm class_weight="balanced" có cải thiện LR mặc định không?
Kết quả có thể tăng hoặc giảm; không điều chỉnh thiết kế theo outer test.

## 03 — Dữ liệu
Bảng physiological68, giữ nguyên schema, checksum, nhãn và 15 subject theo cấu hình đã chạy. Không đưa metadata vào X.

In [2]:
feature_path = ROOT / CONFIG["features"]
if feature_path.is_file():
    assert hashlib.sha256(feature_path.read_bytes()).hexdigest() == CONFIG["feature_sha256"]
for name, expected in PROVENANCE["source_hashes"].items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == expected
assert hashlib.sha256(RESULTS_PATH.read_bytes()).hexdigest() == PROVENANCE["source_hashes"][CONFIG["source_archive"] + "/results.csv"]
print("Đã đối chiếu nguồn và checksum; không huấn luyện lại.")

Đã đối chiếu nguồn và checksum; không huấn luyện lại.


## 04 — Protocol
LOSO 15 người; binary và multiclass tách riêng. Imputer/scaler học trong train. Không tuning trong thí nghiệm này.

## 05 — Cấu hình
`config.yaml` là cấu hình đọc hồ sơ theo mã chung. `training_settings` và `source_config` giữ cấu hình lịch sử với mã nguồn cũ; không sửa manifest để giả như đã train bằng mã mới.

In [3]:
display(CONFIG)

{'experiment_id': 'EXP-011', 'record_kind': 'analysis_of_saved_rerun', 'legacy_run_id': 'THAI-EXP-003', 'source_archive': 'experiments/03_balancing/THAI-EXP-003_LR/r02', 'source_config': 'experiments/03_balancing/THAI-EXP-003_LR/r02/config.yaml', 'source_notebook': 'notebooks/experiments/Thai_EXP-003_working.ipynb', 'features': 'data/features/g2_physiological68_group15_20261107/all_subjects.csv', 'schema': 'data/features/g2_physiological68_group15_20261107/feature_schema.json', 'feature_sha256': '2f2fddd678f333a800f2869d2e78b407934b2c3492190ed70e28d8a9107fd8ba', 'tasks': ['multiclass', 'binary'], 'seed': 42, 'reference_experiment_id': 'EXP-002', 'training_settings': {'protocol': 'notebook_research_new_runs_v1', 'feature_version': 'physiological_68_v1', 'feature_sha256': '2f2fddd678f333a800f2869d2e78b407934b2c3492190ed70e28d8a9107fd8ba', 'source_paths': ['src/wesad/data.py', 'src/wesad/evaluation.py', 'src/wesad/models.py', 'src/wesad/utils.py'], 'source_hash_normalization': 'LF', 'expe

## 06 — Nguồn thực thi
Notebook này đọc kết quả đã lưu, mặc định không train. Code chạy thật và cấu hình nằm tại `notebooks/experiments/Thai_EXP-003_working.ipynb` và `experiments/03_balancing/THAI-EXP-003_LR/r02`. Notebook nguồn giữ code và output của lần chạy cũ; dùng runner chung ở `src/wesad/` cho các baseline. Không đổi tên API/module G3/G4 chỉ để đổi nhãn giai đoạn.

## 07 — Kết quả
Mean và SD (ddof=1) tính theo 15 người; không dùng điểm gộp cửa sổ thay cho mean theo subject.

In [4]:
metrics = pd.read_csv(RESULTS_PATH)
assert len(metrics) == 30 and not metrics.duplicated(["task", "subject_id"]).any()
assert metrics.groupby("task").size().to_dict() == {"binary": 15, "multiclass": 15}
display(metrics)
summary = metrics.groupby("task")[["macro_f1", "balanced_accuracy"]].agg(["mean", "std"])
display(summary)

   subject_id        task model  n_test  macro_f1  balanced_accuracy  seconds
0         S10  multiclass    lr     426  0.270507           0.367257   0.0927
1         S11  multiclass    lr     417  0.551448           0.603570   0.1025
2         S13  multiclass    lr     417  0.592222           0.635360   0.1048
3         S14  multiclass    lr     410  0.411750           0.531273   0.1103
4         S15  multiclass    lr     418  0.689308           0.756199   0.1067
5         S16  multiclass    lr     415  0.336539           0.373156   0.0998
6         S17  multiclass    lr     426  0.673234           0.685758   0.1109
7          S2  multiclass    lr     395  0.458796           0.613394   0.1068
8          S3  multiclass    lr     401  0.681823           0.645103   0.1024
9          S4  multiclass    lr     404  0.765223           0.748718   0.1137
10         S5  multiclass    lr     414  0.770248           0.834894   0.1094
11         S6  multiclass    lr     412  0.812569           0.78

## 08 — Hình minh họa
Hình dựng từ kết quả/dự đoán đã lưu, dùng mã EXP chung. Không huấn luyện lại. Ảnh mã cũ trong thư mục chỉ dùng truy vết.

In [5]:
image_paths = sorted((FOLDER / "figures").glob(CONFIG["experiment_id"] + "_*.png"))
assert len(image_paths) == 5
for image_path in image_paths:
    display(Image(filename=str(image_path)))

<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>


## 09 — Phân tích và đối chứng
Đối chứng chính: EXP-002. Chỉ so cùng task, subject và dữ liệu.

In [6]:
comparison_path = ROOT / CONFIG["source_archive"] / "comparison_by_subject.csv"
if comparison_path.is_file():
    comparison = pd.read_csv(comparison_path)
    display(comparison)
    delta_columns = [c for c in comparison if c.endswith("_delta")]
    if delta_columns:
        display(comparison.groupby("task")[delta_columns].agg(["mean", "std"]))
for task, part in metrics.groupby("task"):
    display(part.nsmallest(3, "macro_f1")[["task", "subject_id", "macro_f1", "balanced_accuracy"]])

   subject_id        task  ...  macro_f1_delta  balanced_accuracy_delta
0         S10  multiclass  ...       -0.019806                -0.011799
1         S11  multiclass  ...       -0.042330                -0.027097
2         S13  multiclass  ...        0.010122                 0.016129
3         S14  multiclass  ...       -0.057757                -0.028193
4         S15  multiclass  ...       -0.018689                -0.014404
5         S16  multiclass  ...       -0.026967                -0.017699
6         S17  multiclass  ...        0.035716                 0.018266
7          S2  multiclass  ...       -0.007331                -0.007610
8          S3  multiclass  ...        0.020641                 0.022816
9          S4  multiclass  ...        0.063422                 0.041026
10         S5  multiclass  ...       -0.025843                -0.017391
11         S6  multiclass  ...        0.025899                 0.020513
12         S7  multiclass  ...       -0.050920                -0

## 10 — Kết luận và nguồn gốc
- binary: Macro-F1 0.810281 ± 0.186783; Balanced Accuracy 0.831881 ± 0.169527.
- multiclass: Macro-F1 0.585266 ± 0.162972; Balanced Accuracy 0.630372 ± 0.142832.

Kết quả biến động giữa người; không kết luận mô hình cuối chỉ từ một hướng. Chọn đặc trưng là nghiên cứu mở rộng sau khi đã quan sát test. Kết quả không phải kiểm định trên tập độc lập mới.

Nguồn cấu hình, checksum và commit được ghi tại `provenance.json`. So sánh train–test của EXP-001/002/011/016 ở `notebooks/04_evaluation/EXP_LR_train_test_comparison.ipynb`.